# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing all data entities using their `@id` fields as per the Croissant specification.

### Dataset Source
The dataset schema is described in Croissant (JSON-LD) and available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, their `@id` fields, and contained fields (columns/variables). To follow Croissant best practices, all references are via `@id`.

In [ ]:
# List all available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print("Available record sets and their fields (by @id):\n")
available_record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id']
    available_record_set_ids.append(rs_id)
    print(f"Record set: {rs_id} (name: {rs.get('name', '<unnamed>')})")
    fields = rs.get('field', [])
    # If only one field, convert to list
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields/COLUMNS:")
    for f in fields:
        f_id = f['@id'] if isinstance(f, dict) else f
        print(f"    - {f_id}")
    print()

## 3. Data Extraction

Load data from available record set(s) into a pandas DataFrame for further analysis. We will refer to record sets and fields using their `@id` value as noted in the overview.

In [ ]:
# For demonstration, extract data from all record sets listed above
dataframes = {}
for record_set_id in available_record_set_ids:
    # Use records() generator (note: will download file if required)
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id} with columns: {df.columns.tolist()}")
    else:
        print(f"No records found for record set: {record_set_id}")

# For this dataset, if there is a main data table, it is likely the only or first record set -- select it
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nExample rows from main record set ({main_record_set_id}):")
    display(main_df.head())
else:
    print("No DataFrames were loaded.")

## 4. Exploratory Data Analysis (EDA)

Now let's perform processing on the data: filtering, normalization, and grouping using only variables referenced by their `@id`.

We'll pick a numeric field (e.g., Age at diagnosis or Interval in months between cancers if present), filter for high values, normalize, and group by a categorical field (e.g., Sex or Tumor Location).

In [ ]:
# User must check which numeric fields are present; let's print out column names again
print(f"Columns in main DataFrame ({main_record_set_id}):")
print(main_df.columns.tolist())

# For illustration, suppose '@age_at_second_crc_diagnosis' is a numeric field @id and '@sex' is categorical
# Replace with the actual @id as present in main_df.columns
# You may use the following to help you select columns:
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    # Heuristics: Find numeric and grouping fields by @id
    if 'age' in col.lower() or 'interval' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower() or 'location' in col.lower() or 'site' in col.lower():
        group_field_id = col

print(f"Selected numeric field for EDA: {numeric_field_id}")
print(f"Selected grouping field for EDA: {group_field_id}")

# Safeguard: If not found, just run on some numeric column
if numeric_field_id is None:
    # Fallback: pick the first numeric-looking column
    for col in main_df.select_dtypes(include=['number']).columns:
        numeric_field_id = col
        break

# Let's proceed if we found a numeric column
if numeric_field_id is not None:
    threshold = main_df[numeric_field_id].mean()  # Filter records above mean as example
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df[[numeric_field_id]].head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by '{group_field_id}':")
        display(grouped_df)
else:
    print("Could not find a numeric field to analyze. Please check dataset columns.")

## 5. Visualization

Visualize the distribution of a numeric variable and group differences, referencing columns by their `@id`. If matplotlib or seaborn is not available, install them as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if grouping field is present
    if group_field_id is not None and group_field_id in main_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- This notebook demonstrated how to access and process a Croissant-compliant dataset using `mlcroissant`, referencing all entities by their `@id` for reproducibility and schema alignment.
- The FAIR² dataset provides detailed clinicopathological and molecular features of a rare cancer survivor cohort.
- Further domain-specific analysis can be performed by exploring additional variables and relationships using the provided DataFrame(s).